# 20 — Detektor subset 25 % dan 50 %: real_only vs oversample vs geometry vs G3
Input: paket kode, dataset regrouped v4, assets, dataset review G3 subset (`review_p025_g3.csv`, `review_p050_g3.csv`) dan output kernel 18 (study subset).
Per subset empat arm dibangun oleh `build_subset_detector_arms.py` (k = min(defisit, accepted G3); oversample/geometry tepat k; G3 urutan terbaik), lalu YOLOv8n 3000 update, eval tiap 250, seed 0–2, semua arm satu subset pada satu GPU dalam satu sesi (p025 GPU0, p050 GPU1). Validation saja; test tidak dibaca. Keputusan pengguna 28 Sep 2026: review G3 saja, arm synthetic_G0 subset ditiadakan.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess
WORK = Path('/kaggle/working/texture_subset_detectors_v1')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=False)
# Kaggle unpacks an uploaded zip into plain files; accept either form, never both.
archives = list(INPUT.rglob('texture_study_code.zip'))
# A previous stage's kernel output also carries a code copy next to study/; only the code dataset counts.
unpacked = [p.parent for p in INPUT.rglob('package_manifest.json')
            if (p.parent/'code/cyclegan/tools/texture_study.py').exists() and not (p.parent/'study').exists()]
if len(archives) + len(unpacked) != 1:
    raise ValueError(f'Attach exactly one texture study code dataset; zip={archives}, unpacked={unpacked}')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            if not (WORK/item.filename).resolve().is_relative_to(WORK.resolve()): raise ValueError('unsafe zip')
        z.extractall(WORK)
else:
    shutil.copytree(unpacked[0], WORK, dirs_exist_ok=True)
    for p in [WORK, *WORK.rglob('*')]: p.chmod(p.stat().st_mode | 0o200)  # /kaggle/input is read-only
import hashlib
manifest = json.loads((WORK/'package_manifest.json').read_text())
bad = [k for k,v in manifest.items() if hashlib.sha256((WORK/k).read_bytes()).hexdigest() != v]
if bad: raise ValueError(f'Package files differ from package_manifest.json: {bad[:5]}')
print('code package verified:', len(manifest), 'files')
HERE = WORK/'code/cyclegan'
ENV = {**os.environ, 'PYTHONPATH':str(HERE), 'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2', 'PYTHONUNBUFFERED':'1'}
def unique(marker):
    hits=sorted({p.parent for p in INPUT.rglob(marker)})
    if len(hits)!=1: raise ValueError(f'Expected one {marker}; found {hits}')
    return hits[0]
DATASET = unique('manifest.csv')
ASSETS = unique('reviewed_masks.json')
ENV.update(MELON_DATASET=str(DATASET),MELON_ASSETS=str(ASSETS))
os.environ.update(ENV)
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=HERE,env=ENV,check=True)
CLI = HERE/'tools/texture_study.py'
import torch
print(torch.__version__,torch.cuda.is_available())


In [ ]:
# ultralytics pinned to the version of the balancing experiment (C) so the trainer is identical.
import importlib.util, importlib.metadata as meta
if importlib.util.find_spec('ultralytics') is None or meta.version('ultralytics') != '8.4.163':
    subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.163'],check=True)
for package in ('torch','torchvision','ultralytics'): print(package, meta.version(package))
if torch.cuda.device_count() < 1: raise RuntimeError('GPU required')
print('GPUs', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


In [ ]:
# Subset study from notebook 18 output (G0/G3 generators, candidates, private keys) + G3-only reviews.
import csv, hashlib
source=unique('study.json')
STUDY=WORK/'study'
shutil.copytree(source,STUDY)
for p in [STUDY,*STUDY.rglob('*')]: p.chmod(p.stat().st_mode|0o200)
REVIEWS={f:unique(f'review_p{f:03d}_g3.csv')/f'review_p{f:03d}_g3.csv' for f in (25,50)}
for f,path in REVIEWS.items(): print(f, path, hashlib.sha256(path.read_bytes()).hexdigest())
ARMS=WORK/'arms'
for f,path in REVIEWS.items():
    run(HERE/'tools/build_subset_detector_arms.py','--study',STUDY,'--dataset',DATASET,'--fraction',f,'--review',path,'--out',ARMS)
    print(json.dumps(json.loads((ARMS/f'p{f:03d}/arms_summary.json').read_text()),indent=1))


In [ ]:
# All arms of one subset on one GPU in one session: p025 on GPU0, p050 on GPU1. Protocol of experiment C.
import shlex, time
SEEDS=[0,1,2]
PLAN={0:25,1:50}
n_gpu=torch.cuda.device_count()
def arm_names(f): return sorted(p.name for p in (ARMS/f'p{f:03d}').iterdir() if (p/'data.yaml').exists())
def chain(gpu,f):
    argv=[sys.executable,str(HERE/'tools/train_detector_arms.py'),'--arms-dir',str(ARMS/f'p{f:03d}'),'--arms',*arm_names(f),
          '--seeds',*map(str,SEEDS),'--out',str(WORK/f'det_p{f:03d}'),'--updates','3000','--eval-every','250','--export']
    env=f"CUDA_VISIBLE_DEVICES={gpu if n_gpu>=2 else 0} PYTHONPATH={shlex.quote(str(HERE))}"
    log=open(WORK/f'detector_p{f:03d}.log','w')
    return subprocess.Popen(env+' '+shlex.join(argv),shell=True,cwd=str(HERE),stdout=log,stderr=subprocess.STDOUT)
started=time.time()
if n_gpu>=2: codes=[p.wait() for p in [chain(g,f) for g,f in PLAN.items()]]
else: codes=[chain(0,f).wait() for f in PLAN.values()]
print('return codes',codes,'wall minutes',round((time.time()-started)/60,1))
for f in PLAN.values(): print(f'--- p{f:03d}'); print((WORK/f'detector_p{f:03d}.log').read_text()[-1500:])
if any(codes): raise SystemExit('detector training failed')


In [ ]:
# Validation summaries and paired seed deltas; keep the output small.
import statistics
OUT=WORK/'result'; OUT.mkdir()
report={}
for f in PLAN.values():
    part=json.loads((WORK/f'det_p{f:03d}'/'results.json').read_text())
    shutil.copytree(WORK/f'det_p{f:03d}',OUT/f'p{f:03d}',ignore=shutil.ignore_patterns('weights.pt','last*.pt'))
    shutil.copy(ARMS/f'p{f:03d}'/'arms_summary.json',OUT/f'p{f:03d}'/'arms_summary.json')
    for name in (ARMS/f'p{f:03d}').glob('ranking_*.csv'): shutil.copy(name,OUT/f'p{f:03d}'/name.name)
    for arm in arm_names(f): shutil.copy(ARMS/f'p{f:03d}'/arm/'arm_manifest.json',OUT/f'p{f:03d}'/f'{arm}_manifest.json')
    rows={}
    for key,table in (('selected',part['selected']),('final',part['results'])):
        summ={}
        for arm in arm_names(f):
            v=[table[f'{arm}@{s}'] for s in SEEDS]
            summ[arm]=dict(map50_95=statistics.fmean(x['map50_95'] for x in v),sd=statistics.pstdev(x['map50_95'] for x in v),
                           ap_unripe=statistics.fmean(x['per_class']['unripe']['ap50_95'] for x in v),
                           ap_ripe=statistics.fmean(x['per_class']['ripe']['ap50_95'] for x in v),
                           recall_unripe=statistics.fmean(x['recall'][1] for x in v))
        syn=next(a for a in arm_names(f) if a.startswith('synthetic'))
        deltas={ref:[table[f'{syn}@{s}']['map50_95']-table[f'{ref}@{s}']['map50_95'] for s in SEEDS] for ref in arm_names(f) if ref!=syn}
        rows[key]=dict(summary=summ,paired_delta_map50_95=deltas)
    report[f]=rows
    print(f'p{f:03d}',json.dumps(rows['selected'],indent=1))
(OUT/'results.json').write_text(json.dumps(dict(report=report,seeds=SEEDS,updates=3000,eval_every=250,split='valid',test_used=False),indent=2)+'\n')
for d in (STUDY,ARMS,WORK/'code',*[WORK/f'det_p{f:03d}' for f in PLAN.values()]): shutil.rmtree(d,ignore_errors=True)
print(sum(1 for p in WORK.rglob('*') if p.is_file()),'output files')
